# Phase 5 – Evaluation

**CRISP-DM Step 5 of 5**

Goals:
- Validate model assumptions (residuals, normality, homoscedasticity)
- Interpret fixed and random effects
- Compare model performance (AIC/BIC, LRT)
- Run robustness checks (alternative outcome, subsamples, alternative grouping)
- Assess against the business success criteria from Phase 1
- Summarise findings and limitations

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.data.loader import load_processed
from src.models.multilevel import (
    fit_null_model, fit_linear_mixed, fit_ordinal_logit,
    compare_models, plot_residuals, random_effects_caterpillar, plot_fixed_effects,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 5.1 Load Data and Refit Best Model

We refit M2 (RE count + controls, random intercepts by country) as the primary model for diagnostics.

In [ ]:
df = load_processed('sme_model_ready.parquet')

OUTCOME   = 'turnover_change_ord'
RE_VAR    = 're_action_count'
CONTROLS  = ['firm_size_ord', 'firm_age_ord', 'turnover_bracket_ord']
GROUP     = 'country_label'      # Level 2 grouping for primary model
GROUP_MACRO = 'macro_region'     # Level 3

df_main = df.dropna(subset=[OUTCOME, RE_VAR] + CONTROLS + [GROUP, GROUP_MACRO]).reset_index(drop=True)
print(f"Analysis sample: {len(df_main):,} firms across "
      f"{df_main[GROUP].nunique()} countries / "
      f"{df_main[GROUP_MACRO].nunique()} macro-regions")

# Refit best model: M2 (RE + controls, RI by country)
best_model = fit_linear_mixed(
    df_main,
    outcome=OUTCOME,
    fixed_effects=[RE_VAR] + CONTROLS,
    group=GROUP,
    model_name='M2_best',
)

## 5.2 Residual Diagnostics

In [ ]:
fig = plot_residuals(best_model)
plt.savefig('../reports/figures/residual_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Shapiro-Wilk normality test on a random sample
resid = best_model.result.resid
sample = resid.sample(min(len(resid), 5000), random_state=42)
stat, p = stats.shapiro(sample)
print(f"Shapiro-Wilk: W = {stat:.4f}, p = {p:.4f}")
print("Note: with large N, Shapiro-Wilk is very sensitive; rely primarily on Q-Q plot.")

## 5.3 Fixed Effects Interpretation

In [ ]:
result = best_model.result
coef_table = pd.DataFrame({
    'coef':     result.fe_params,
    'se':       result.bse_fe,
    'z':        result.fe_params / result.bse_fe,
    'p_value':  result.pvalues,
    'ci_lower': result.conf_int().iloc[:, 0],
    'ci_upper': result.conf_int().iloc[:, 1],
}).drop('Intercept', errors='ignore').round(4)

coef_table['sig'] = coef_table['p_value'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
)
display(coef_table)

In [ ]:
fig = plot_fixed_effects(best_model)
plt.savefig('../reports/figures/fixed_effects_best.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.4 Random Effects Distribution

In [ ]:
fig = random_effects_caterpillar(best_model)
plt.savefig('../reports/figures/random_effects_caterpillar.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nICC (country level): {best_model.icc:.4f} ({best_model.icc*100:.1f}% of variance)")

## 5.5 RE Effect Visualisation

In [ ]:
# Mean turnover change score by RE action count
re_outcome = df_main.groupby(RE_VAR)[OUTCOME].agg(['mean', 'sem', 'count']).reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(
    re_outcome[RE_VAR], re_outcome['mean'],
    yerr=1.96 * re_outcome['sem'],
    fmt='o-', color='steelblue', capsize=4, linewidth=2,
)
ax.set_xlabel('Number of resource efficiency actions')
ax.set_ylabel('Mean turnover change score (1=declined, 4=grew ≥10%)')
ax.set_title('RE Action Count vs. Turnover Change (mean ± 95% CI)')
plt.tight_layout()

## 5.6 Robustness Checks

### 5.6.1 Alternative outcome: cost impact (q3)

In [ ]:
df_cost = df.dropna(subset=['cost_impact_ord', RE_VAR] + CONTROLS).reset_index(drop=True)
print(f"Cost impact sample: {len(df_cost):,} firms (requires \u22651 RE action)")

m_cost = fit_linear_mixed(
    df_cost,
    outcome='cost_impact_ord',
    fixed_effects=[RE_VAR] + CONTROLS,
    group=GROUP,
    model_name='robustness_cost_impact',
)
print(m_cost.summary())

### 5.6.2 Alternative outcome: employee change

In [ ]:
df_emp = df.dropna(subset=['emp_change_ord', RE_VAR] + CONTROLS).reset_index(drop=True)

m_emp = fit_linear_mixed(
    df_emp,
    outcome='emp_change_ord',
    fixed_effects=[RE_VAR] + CONTROLS,
    group=GROUP,
    model_name='robustness_emp_change',
)
# Print RE coefficient only
print(f"Employee change ~ RE count:  coef = {m_emp.result.fe_params[RE_VAR]:.4f}, "
      f"p = {m_emp.result.pvalues[RE_VAR]:.4f}")

### 5.6.3 Micro firms only (scr10 == 1)

In [ ]:
df_micro = df_main[df_main['firm_size_ord'] == 1].copy()
print(f"Micro firms: {len(df_micro):,}")

if len(df_micro) > 500 and df_micro[GROUP].nunique() >= 5:
    m_micro = fit_linear_mixed(
        df_micro,
        outcome=OUTCOME,
        fixed_effects=[RE_VAR] + ['firm_age_ord', 'turnover_bracket_ord'],
        group=GROUP,
        model_name='robustness_micro_only',
    )
    print(f"Micro only: RE coef = {m_micro.result.fe_params[RE_VAR]:.4f}, "
          f"p = {m_micro.result.pvalues[RE_VAR]:.4f}")

### 5.6.4 Grouped by EU macro-region (alternative Level 3 grouping)

In [ ]:
# Fit same model using macro_region as the grouping unit instead of country
# (checks whether the RE effect holds at a coarser geographic level)
import statsmodels.formula.api as smf

fe_str = RE_VAR + ' + ' + ' + '.join(CONTROLS)
m_macro = smf.mixedlm(
    f'{OUTCOME} ~ {fe_str}',
    data=df_main,
    groups=df_main[GROUP_MACRO],
).fit(reml=True)

re_coef_macro = m_macro.fe_params[RE_VAR]
re_pval_macro = m_macro.pvalues[RE_VAR]
var_macro = float(m_macro.cov_re.iloc[0, 0])
icc_macro = var_macro / (var_macro + m_macro.scale)

print(f"Grouped by macro-region: RE coef = {re_coef_macro:.4f}, p = {re_pval_macro:.4f}")
print(f"ICC (macro-region): {icc_macro:.4f}")

### 5.6.5 Ordinal logit sensitivity (no random effects)

In [ ]:
m_ord = fit_ordinal_logit(
    df_main,
    outcome=OUTCOME,
    predictors=[RE_VAR] + CONTROLS,
    model_name='sensitivity_ordinal_logit',
)
print(m_ord.summary())

In [ ]:
from src.models.multilevel import breusch_pagan_test

# Breusch-Pagan heteroscedasticity test
print("\n--- Breusch-Pagan Test (best_model) ---")
bp = breusch_pagan_test(best_model)
print()
print("Interpretation:")
print("  If p < 0.05: heteroscedastic residuals -> OLS SEs are invalid.")
print("  For an ordinal outcome modelled as continuous, some")
print("  heteroscedasticity is expected.")
print()

# Shapiro-Wilk on a random subsample (test is unreliable at large N)
from scipy import stats as scipy_stats
resid2 = np.array(best_model.result.resid)
rng = np.random.default_rng(42)
idx = rng.choice(len(resid2), size=min(5000, len(resid2)), replace=False)
stat_sw, p_sw = scipy_stats.shapiro(resid2[idx])
print(f"Shapiro-Wilk (n={len(idx):,} subsample): W={stat_sw:.4f}, p={p_sw:.4f}")
print("Note: Shapiro-Wilk is very sensitive at N > 2,000; a significant")
print("result here means minor non-normality, not necessarily a major violation.")

In [ ]:
from src.models.multilevel import cooks_distance_plot

# Approximated Cook's distance using marginal OLS model
# Flags observations with undue influence on fixed-effect estimates.
fig = cooks_distance_plot(best_model, threshold=4.0)
plt.show()

# Random effects normality check (Q-Q of BLUPs)
re_vals = np.array([v.iloc[0] for v in best_model.result.random_effects.values()])
fig2, ax = plt.subplots(figsize=(5, 5))
from scipy import stats as scipy_stats
scipy_stats.probplot(re_vals, dist="norm", plot=ax)
ax.set_title("Q-Q plot of country random intercepts (BLUPs)\n"
             "Points should follow the diagonal if normality holds")
plt.tight_layout()
plt.show()
print(f"Country BLUPs: n={len(re_vals)}, mean={re_vals.mean():.3f}, "
      f"SD={re_vals.std():.3f}")

In [ ]:
# Refit models with ML for valid AIC/BIC comparison
models_ml = [
    fit_null_model(df_main, OUTCOME, GROUP, reml=False),
    fit_linear_mixed(df_main, OUTCOME, [RE_VAR], GROUP, reml=False, model_name='M1_re'),
    fit_linear_mixed(df_main, OUTCOME, [RE_VAR] + CONTROLS, GROUP, reml=False, model_name='M2_re_controls'),
]
comparison = compare_models(models_ml)
display(comparison)

## 5.8 Success Criteria Assessment

In [ ]:
icc_val = best_model.icc
re_coef = best_model.result.fe_params.get(RE_VAR, np.nan)
re_pval = best_model.result.pvalues.get(RE_VAR, np.nan)

criteria = pd.DataFrame([
    {'Criterion': 'ICC > 0.05 at country level',
     'Target': '>0.05',
     'Result': f'{icc_val:.4f}',
     'Pass': '✅' if icc_val > 0.05 else '❌'},
    {'Criterion': 'RE action count: significant positive coef',
     'Target': 'p < 0.05, coef > 0',
     'Result': f'coef={re_coef:.4f}, p={re_pval:.4f}',
     'Pass': '✅' if (re_pval < 0.05 and re_coef > 0) else '❌'},
    {'Criterion': 'Models converge without warnings',
     'Target': 'True',
     'Result': 'Check model output',
     'Pass': '?'},
    {'Criterion': 'Processed dataset saved',
     'Target': 'sme_model_ready.parquet',
     'Result': 'Yes (Phase 3)',
     'Pass': '✅'},
])

display(criteria)

## 5.9 Final Findings Summary

Fill in after running the full pipeline.

### Main Findings

1. **ICC**: X% of variance in turnover change is attributable to the country level, justifying HLM.
2. **Resource Efficiency Effect**: A one-unit increase in RE action count (0–10) is associated with a β = X change in the ordinal turnover change score (p = ?, 95% CI: [?, ?]).
3. **Firm Controls**: Larger firms and firms with higher baseline turnover show higher turnover growth.
4. **Random Effects**: Country-level random intercepts show substantial variation in turnover outcomes across EU-27.
5. **Best Model**: M2 (RE + controls, RI by country) with AIC = ?.

### Limitations

- **Cross-sectional design**: cannot establish causality; reverse causality possible (growing firms may invest more in RE)
- **Self-reported data**: turnover change and RE actions are both self-reported (common method variance)
- **Ordinal outcome treated as quasi-continuous**: linear mixed model assumption; sensitivity check via ordinal logit
- **Structural missing**: firms with no RE actions lack Q3/Q4 data, limiting sub-analyses
- **No longitudinal follow-up**: we cannot observe whether RE investments pay off over time

### Next Steps

- Apply proper ordinal mixed model (R: `ordinal::clmm`) via `rpy2` if available
- Explore mediation analysis: RE actions → cost reduction → turnover growth
- Match survey respondents to external firm databases for objective financial outcomes
- Replicate with weighted estimates using `w1_sme` for population-representative figures